# Task 1: News Topic Classifier Using BERT
**DevelopersHub Corporation — AI/ML Engineering Internship**

## Objective
Fine-tune `bert-base-uncased` on the AG News dataset to classify news headlines into 4 topic categories: **World, Sports, Business, Sci/Tech**.

## Skills
- NLP using Transformers (Hugging Face)
- Transfer learning & fine-tuning
- Evaluation metrics (Accuracy, F1)
- Streamlit deployment

---

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────
# Uncomment if needed:
# !pip install transformers datasets torch torchvision seaborn plotly streamlit
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Load & Explore the AG News Dataset

In [ ]:
ds = load_dataset("ag_news")
print(ds)
print("\nSample entries:")
for i in range(3):
    print(f"  Label={ds['train'][i]['label']}  Text: {ds['train'][i]['text'][:80]}…")

In [ ]:
# Convert to DataFrames and subsample
TRAIN_N, TEST_N = 8000, 2000
train_df = pd.DataFrame(ds["train"]).sample(TRAIN_N, random_state=42)
test_df  = pd.DataFrame(ds["test"]).sample(TEST_N,  random_state=42)

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

# Visualise label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for df_, ax, title in [(train_df, axes[0], "Train"), (test_df, axes[1], "Test")]:
    counts = df_["label"].value_counts().sort_index()
    ax.bar([LABEL_NAMES[i] for i in counts.index], counts.values, color=["#3498db","#2ecc71","#e67e22","#9b59b6"])
    ax.set_title(f"AG News – {title} Set")
    ax.set_ylabel("Count")
plt.tight_layout(); plt.show()
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

## 3. Tokenisation & Dataset Class

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128
tokenizer  = BertTokenizerFast.from_pretrained(MODEL_NAME)

class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label": torch.tensor(self.labels[idx], dtype=torch.long)}

# Verify tokenisation
sample = tokenizer("AI researchers develop new breakthrough in large language models.",
                    max_length=32, padding="max_length", truncation=True)
print("Token IDs (first 15):", sample["input_ids"][:15])
print("Decoded:", tokenizer.decode(sample["input_ids"][:15]))

## 4. Fine-Tune BERT

In [ ]:
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = AGNewsDataset(train_df["text"].tolist(), train_df["label"].tolist(), tokenizer, MAX_LEN)
test_ds  = AGNewsDataset(test_df["text"].tolist(),  test_df["label"].tolist(),  tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4).to(DEVICE)
total_steps = len(train_loader) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LR, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, total_steps//10, total_steps)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Training on: {DEVICE}")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train(); total_loss, preds_all, labels_all = 0, [], []
    for batch in loader:
        ids, mask, labels = (batch[k].to(DEVICE) for k in ["input_ids","attention_mask","label"])
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        preds_all.extend(out.logits.argmax(-1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return total_loss/len(loader), accuracy_score(labels_all,preds_all), f1_score(labels_all,preds_all,average="weighted")

def eval_epoch(model, loader):
    model.eval(); total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, labels = (batch[k].to(DEVICE) for k in ["input_ids","attention_mask","label"])
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            total_loss += out.loss.item()
            preds_all.extend(out.logits.argmax(-1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return total_loss/len(loader), accuracy_score(labels_all,preds_all), f1_score(labels_all,preds_all,average="weighted"), preds_all, labels_all

history = {k:[] for k in ["train_loss","val_loss","train_acc","val_acc","train_f1","val_f1"]}

for epoch in range(1, EPOCHS+1):
    tr = train_epoch(model, train_loader, optimizer, scheduler)
    va = eval_epoch(model, test_loader)
    for k,v in zip(history, [tr[0],va[0],tr[1],va[1],tr[2],va[2]]): history[k].append(v)
    print(f"Epoch {epoch}/{EPOCHS}  loss={tr[0]:.4f}/{va[0]:.4f}  acc={tr[1]:.4f}/{va[1]:.4f}  f1={tr[2]:.4f}/{va[2]:.4f}")

## 5. Evaluation & Visualisations

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = range(1, EPOCHS+1)
for ax, keys, title in zip(axes, [("train_loss","val_loss"),("train_acc","val_acc"),("train_f1","val_f1")], ["Loss","Accuracy","F1"]):
    ax.plot(epochs, history[keys[0]], "o--", label="Train"); ax.plot(epochs, history[keys[1]], "o-", label="Val")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("BERT Fine-tuning – AG News", fontsize=14, fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix
_, _, _, final_preds, final_labels = eval_epoch(model, test_loader)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(final_labels, final_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
axes[0].set_title("Confusion Matrix"); axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")

# Per-class F1
from sklearn.metrics import classification_report
report = classification_report(final_labels, final_preds, target_names=LABEL_NAMES, output_dict=True)
per_class_f1 = {k: report[k]["f1-score"] for k in LABEL_NAMES}
axes[1].barh(list(per_class_f1.keys()), list(per_class_f1.values()), color=["#3498db","#2ecc71","#e67e22","#9b59b6"])
axes[1].set_xlim(0, 1.1); axes[1].set_title("Per-Class F1 Score")
for i, v in enumerate(per_class_f1.values()): axes[1].text(v+0.01, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()

print(classification_report(final_labels, final_preds, target_names=LABEL_NAMES))

## 6. Save Model & Summary

In [ ]:
torch.save({"model_state": model.state_dict(), "config": MODEL_NAME}, "bert_ag_news.pt")
tokenizer.save_pretrained("tokenizer/")
print("Model saved: bert_ag_news.pt")
print("Tokenizer saved: tokenizer/")

final_acc = accuracy_score(final_labels, final_preds)
final_f1  = f1_score(final_labels, final_preds, average="weighted")

print(f"\n{'='*50}")
print("FINAL RESULTS")
print(f"  Accuracy (weighted) : {final_acc:.4f} ({final_acc*100:.1f}%)")
print(f"  F1 Score (weighted) : {final_f1:.4f}")
print(f"  Model               : {MODEL_NAME} fine-tuned on AG News")
print(f"  Training samples    : {TRAIN_N} | Test samples: {TEST_N}")
print(f"  Epochs              : {EPOCHS} | LR: {LR}")
print(f"{'='*50}")

print("\n📌 Key Insights:")
print("  • BERT excels at news classification due to its bidirectional context understanding")
print("  • Transfer learning allows high accuracy with only 8K training samples")
print("  • Sports and Sci/Tech tend to have the highest per-class F1 scores")
print("  • Business and World can overlap, creating harder classification boundaries")

## 7. Deployment Notes

Run the Streamlit app for live predictions:
```bash
streamlit run app.py
```

The app allows entering any news headline and returns the predicted category with confidence scores.